<a href="https://colab.research.google.com/github/9lenrg/medley_vox/blob/main/MedleyVox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab Inference for MedleyVox

Medley Vox is a [dataset for testing algorithms for separating multiple singers](https://arxiv.org/pdf/2211.07302) within a single music track. Also, the [authors of Medley Vox](https://github.com/jeonchangbin49/MedleyVox) proposed a neural network architecture for separating singers. However, unfortunately, they did not publish the weights. Later, their training process was [repeated by Cyru5](https://huggingface.co/Cyru5/MedleyVox/tree/main), who trained several models and published the weights in the public domain. Now this WebUI is created to use the trained models and weights for inference. Here are some precautions:
1. Put the [downloaded models](https://huggingface.co/Cyru5/MedleyVox) in the 'checkpoints' folder in folder format, with each model folder containing a model file (.pth) and its corresponding configuration file (.json).
2. If you use overlapadd and the choice of model is 'w2v' or 'w2v_chunk', you need to download the pretrained model [xlsr_53_56k.pt](https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr_53_56k.pt) and put it in the 'pretrained' folder.
3. At present, the audio output sampling rate supported by the model is 24000kHz and cannot be changed. To solve this, you can use [AudioSR](https://github.com/haoheliu/versatile_audio_super_resolution), [Apollo](https://github.com/JusperLee/Apollo), or [Music Source Separation Training](https://github.com/ZFTurbo/Music-Source-Separation-Training) for audio super-resolution.
4. When using WebUI on cloud platforms or Colab, please place the audio to be processed in the 'inputs' folder, and the processing results will be stored in the 'results' folder. The 'Select folder' and 'Open folder' buttons are invalid in the cloud.
5. If the input is too long, it may be impossible to inference due to lack of VRAM. In that case, use 'use_overlapadd'. Among the 'use_overlapadd' options, "ola", "ola_norm", and "w2v" all work well. Use w2v_chunk or sf_chunk if these fail or as desired. You can also try experimenting with 'vad_method' options spec and webrtc when using either of the "_chunk" methods. Chunking has proven to be very useful therefore it is on by default.

# Initialize environment

In [1]:
#@title Clone repository and install requirements
#@markdown # Clone repository and install requirements
#@markdown

!nvidia-smi
!git clone https://github.com/SUC-DriverOld/MedleyVox-Inference-WebUI
%cd /content/MedleyVox-Inference-WebUI
!python -m pip install --upgrade pip==24.0 setuptools
!python -m pip install -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu124
!mkdir -p inputs
!mkdir -p results

!mkdir -p "/content/MedleyVox-Inference-WebUI/checkpoint/vocals_238"
%cd "/content/MedleyVox-Inference-WebUI/checkpoint/vocals_238"
!wget https://huggingface.co/Cyru5/MedleyVox/resolve/main/vocals%20238/vocals.pth
!wget https://huggingface.co/Cyru5/MedleyVox/resolve/main/vocals%20238/vocals.json
!mkdir -p "/content/MedleyVox-Inference-WebUI/checkpoint/multi_singing_librispeech_138"
%cd "/content/MedleyVox-Inference-WebUI/checkpoint/multi_singing_librispeech_138"
!wget https://huggingface.co/Cyru5/MedleyVox/resolve/main/multi_singing_librispeech_138/vocals.pth
!wget https://huggingface.co/Cyru5/MedleyVox/resolve/main/multi_singing_librispeech_138/vocals.json
!mkdir -p "/content/MedleyVox-Inference-WebUI/checkpoint/singing_librispeech_ft_iSRNet"
%cd "/content/MedleyVox-Inference-WebUI/checkpoint/singing_librispeech_ft_iSRNet"
!wget https://huggingface.co/Cyru5/MedleyVox/resolve/main/singing_librispeech_ft_iSRNet/vocals.pth
!wget https://huggingface.co/Cyru5/MedleyVox/resolve/main/singing_librispeech_ft_iSRNet/vocals.json
!mkdir -p "/content/MedleyVox-Inference-WebUI/pretrained"
%cd "/content/MedleyVox-Inference-WebUI/pretrained"
!wget https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr_53_56k.pt
%cd /content/MedleyVox-Inference-WebUI

Mon Mar 16 06:42:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Inference

### Place the audio to be processed in the 'inputs' folder, and the processing results will be stored in the 'results' folder. There are two ways to tun inference: use WebUI or use command line.

- Use WebUI: Run the WebUI startup code block and then access the WebUI through the public link.
- Use command line: Select appropriate inference parameters and run the the command line code block.

### Explanation of reasoning parameters. For more infrmation, refer to `inference.py`.

- `Model name`: Select which model you want to use.
- `Use overlapadd`: Use overlapadd functions, ola, ola_norm, w2v will work with ola_window_len, ola_hop_len argugments. w2v_chunk and sf_chunk is chunk-wise processing based on VAD, so you have to specify the vad_method args. If you use sf_chunk (spectral_featrues_chunk), you also need to specify spectral_features.
- `Separate storage`: Save results in separate folders with the same name as the input file.
- `Output format`: Select the output format of the results.
- `VAD method`: What method do you want to use for 'voice activity detection (vad) -- split chunks -- processing. Only valid when 'w2v_chunk' or 'sf_chunk' for args.use_overlapadd.
- `Spectral features`: What spectral feature do you want to use in correlation calc in speaker assignment (only valid when using sf_chunk)
- `OLA window length`: OLA window size in [sec], only valid when using ola or ola_norm. Set 0 to use the default value (None).
- `OLA hop length`: OLA hop size in [sec], only valid when using ola or ola_norm. Set 0 to use the default value (None).
- `Wav2Vec nth layer output`: Wav2Vec nth layer output, only valid when using w2v or w2v_chunk. For example: 0 1 2 3, default: 0
- `Use EMA model`: Use EMA model or online model? Only vaind when args.ema it True (model trained with EMA).
- `Mix consistent output`: Only valid when the model is trained with mixture_consistency loss.
- `Reorder chunks`: OLA reorder chunks. Only valid when using ola or ola_norm.
- `Skip error files`: Skip error files while separating instead of stopping.

If the input is too long, it may be impossible to inference due to lack of VRAM. In that case, use `use_overlapadd`. Among the `use_overlapadd` options, "ola", "ola_norm", and "w2v" all work well. Use w2v_chunk or sf_chunk if these fail or as desired. You can also try experimenting with `vad_method` options spec and webrtc when using either of the "_chunk" methods. Chunking has proven to be very useful therefore it is on by default.

In [6]:
!pip install --upgrade librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.7/260.7 kB 5.9 MB/s eta 0:00:00
DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
  Attempting uninstall: librosa
    Found existing installation: librosa 0.9.1
    Uninstalling librosa-0.9.1:
      Successfully uninstalled librosa-0.9.1


In [25]:
!cp -r /content/MedleyVox-Inference-WebUI/checkpoint /content/MedleyVox-Inference-WebUI/checkpoints

In [27]:
import os
import shutil
import glob

print("Merapikan struktur folder model...")
base = "/content/MedleyVox-Inference-WebUI"

# 1. Siapkan folder target yang benar
correct_path1 = os.path.join(base, "checkpoint", "multi_singing_librispeech_138")
correct_path2 = os.path.join(base, "checkpoints", "multi_singing_librispeech_138")
os.makedirs(correct_path1, exist_ok=True)
os.makedirs(correct_path2, exist_ok=True)

# 2. Cari dan pindahkan file vocals.pth ke tempat yang benar
for pth in glob.glob(f"{base}/**/*.pth", recursive=True):
    if "xlsr" not in pth and "vocals.pth" in pth:
        shutil.copy(pth, os.path.join(correct_path1, "vocals.pth"))
        shutil.copy(pth, os.path.join(correct_path2, "vocals.pth"))
        break

# 3. Cari dan pindahkan file vocals.json ke tempat yang benar
for jsn in glob.glob(f"{base}/**/*.json", recursive=True):
    if "vocals.json" in jsn:
        shutil.copy(jsn, os.path.join(correct_path1, "vocals.json"))
        shutil.copy(jsn, os.path.join(correct_path2, "vocals.json"))
        break

# 4. Hapus folder "checkpoint" nyasar yang muncul di menu Gradio
bad_dir_1 = os.path.join(base, "checkpoint", "checkpoint")
bad_dir_2 = os.path.join(base, "checkpoints", "checkpoint")
if os.path.exists(bad_dir_1): shutil.rmtree(bad_dir_1)
if os.path.exists(bad_dir_2): shutil.rmtree(bad_dir_2)

print("Selesai! Susunan folder sudah rapi sempurna.")

Merapikan struktur folder model...
Selesai! Susunan folder sudah rapi sempurna.


In [29]:
import os
import re

folder_path = "/usr/local/lib/python3.12/dist-packages/fairseq"
patched = 0

print("Memulai pembersihan paksa fairseq...")

for root, _, files in os.walk(folder_path):
    for file in files:
        if file.endswith(".py"):
            file_path = os.path.join(root, file)
            # errors="ignore" akan memaksa script mengabaikan karakter teks yang rusak
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                content = f.read()

            # Cari format lama dan ubah ke default_factory
            new_content = re.sub(r'(\w+)\s*:\s*([A-Z][A-Za-z0-9_]*)\s*=\s*\2\(\)', r'\1: \2 = field(default_factory=\2)', content)

            if new_content != content:
                if "from dataclasses import field" not in new_content:
                    new_content = "from dataclasses import field\n" + new_content
                with open(file_path, "w", encoding="utf-8") as f:
                    f.write(new_content)
                print(f"Berhasil menambal: {file}")
                patched += 1

print(f"Selesai! {patched} file berpenyakit sudah disembuhkan secara paksa.")

Memulai pembersihan paksa fairseq...
Selesai! 0 file berpenyakit sudah disembuhkan secara paksa.


In [31]:
import os
import re

folder_path = "/usr/local/lib/python3.12/dist-packages/fairseq"
patched = 0

print("Mengaktifkan Mode Dewa: Menghapus semua bug konfigurasi...")

for root, _, files in os.walk(folder_path):
    for file in files:
        if file.endswith(".py"):
            file_path = os.path.join(root, file)
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                content = f.read()

            # Membedah secara paksa semua class yang berakhiran "Config"
            # Contoh: = QuantNoiseConfig(apa_saja) -> = field(default_factory=lambda: QuantNoiseConfig(apa_saja))
            new_content = re.sub(
                r'=\s*([A-Z][a-zA-Z0-9_]*Config)\(([^)]*)\)',
                r'= field(default_factory=lambda: \1(\2))',
                content
            )

            if new_content != content:
                if "from dataclasses import field" not in new_content:
                    new_content = "from dataclasses import field\n" + new_content
                with open(file_path, "w", encoding="utf-8") as f:
                    f.write(new_content)
                print(f"Berhasil menambal: {file}")
                patched += 1

print(f"Selesai! {patched} file berpenyakit berhasil diamputasi.")

Mengaktifkan Mode Dewa: Menghapus semua bug konfigurasi...
Berhasil menambal: initialize.py
Berhasil menambal: utils.py
Berhasil menambal: data_cfg.py
Berhasil menambal: speech_to_text.py
Berhasil menambal: speech_to_speech.py
Berhasil menambal: speech_ulm_task.py
Berhasil menambal: hf_gpt2.py
Berhasil menambal: transformer_config.py
Berhasil menambal: adaptive_span_model_wrapper.py
Berhasil menambal: prepare_dataset.py
Berhasil menambal: sample.py
Berhasil menambal: cont_metrics.py
Berhasil menambal: fb_convert_beit_cp.py
Berhasil menambal: data2vec2.py
Berhasil menambal: data2vec_text.py
Berhasil menambal: data2vec_vision.py
Berhasil menambal: data2vec_audio.py
Berhasil menambal: base.py
Berhasil menambal: wav2vec_featurize.py
Berhasil menambal: transformer_xl_model.py
Berhasil menambal: multi_modality_compound.py
Berhasil menambal: speech_text_joint.py
Berhasil menambal: speech_text_denoise_pretrain.py
Selesai! 23 file berpenyakit berhasil diamputasi.


In [33]:
import os

file_path = "/usr/lib/python3/dist-packages/pkg_resources/__init__.py"

if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Kita kelabuhi sistemnya agar tidak mencari ImpImporter lagi
    new_content = content.replace("pkgutil.ImpImporter", "getattr(pkgutil, 'ImpImporter', type(None))")

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(new_content)
    print("Berhasil! Minion bug pkg_resources sudah dibasmi.")
else:
    print("Aman, file tidak ditemukan.")

Berhasil! Minion bug pkg_resources sudah dibasmi.


In [35]:
!rm -rf /usr/lib/python3/dist-packages/pkg_resources
print("Sarang bug bawaan pabrik sudah dihanguskan! 🔥")

Sarang bug bawaan pabrik sudah dihanguskan! 🔥


In [37]:
!pip install --force-reinstall setuptools
print("Sistem yang sehat sudah dikembalikan! 🏥")

  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
Using cached setuptools-82.0.1-py3-none-any.whl (1.0 MB)
DEPRECATION: omegaconf 2.0.6 has a non-standard dependency specifier PyYAML>=5.1.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of omegaconf or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
  Attempting uninstall: setuptools
    Found existing installation: setuptools 82.0.1
    Uninstalling setuptools-82.0.1:
      Successfully uninstalled setuptools-82.0.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torchvision 0.25.0+cu128 requires torch==2.10.0, but you have torch 2.5.1+cu124 which is incompatible

Sistem yang sehat sudah dikembalikan! 🏥


In [2]:
import os
import re

file_path = "/usr/local/lib/python3.12/dist-packages/webrtcvad.py"

if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Mematikan import pkg_resources
    content = content.replace("import pkg_resources", "# import pkg_resources")

    # Mengganti kode pengecekan versi yang ribet dengan teks manual
    content = re.sub(
        r"__version__\s*=\s*pkg_resources\.get_distribution\([^)]*\)\.version",
        '__version__ = "2.0.10"',
        content
    )

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)
    print("Operasi sukses! webrtcvad sudah bungkam dan tidak butuh pkg_resources lagi. 🎯")
else:
    print("File tidak ditemukan, sepertinya sudah aman.")

Operasi sukses! webrtcvad sudah bungkam dan tidak butuh pkg_resources lagi. 🎯


In [3]:
#@title Run inference in WebUI
#@markdown # Run inference in WebUI
#@markdown

#@markdown

#@markdown Language Setting
language = "English" #@param ["English", "简体中文"]

import os
language_dict = {"English": "en_US", "简体中文": "zh_CN"}
os.environ["LANGUAGE"] = language_dict[language]

%cd /content/MedleyVox-Inference-WebUI
!python webui.py -s

/content/MedleyVox-Inference-WebUI
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://e857aa15c9a502c929.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Inference started... Click 'Forced stop inference' to stop the process.
Inference process started, PID: 22049.
/usr/bin/python3 inference.py --target "vocals" --exp_name "multi_singing_librispeech_138" --model_dir "checkpoints" --use_gpu y --use_overlapadd ola --vad_method spec --spectral_features mfcc --w2v_ckpt_dir pretrained --w2v_nth_layer_output 0 --use_ema_model y --mix_consistent_out y --reorder_chunks y --skip_error y --separate_storage n --output_format wav --inference_data_dir "temp" --results_save_dir "results/"
2026-03-16 07:52:13.708850: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [10]:
!pip uninstall -y fairseq
!pip install git+https://github.com/facebookresearch/fairseq.git

Found existing installation: fairseq 0.12.2
Uninstalling fairseq-0.12.2:
  Successfully uninstalled fairseq-0.12.2
  Cloning https://github.com/facebookresearch/fairseq.git to /tmp/pip-req-build-c9zdcj_3
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/fairseq.git /tmp/pip-req-build-c9zdcj_3
  Resolved https://github.com/facebookresearch/fairseq.git to commit 3d262bb25690e4eb2e7d3c1309b1e9c406ca4b99
  Running command git submodule update --init --recursive -q
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for fairseq: filename=fairseq-0.12.2-cp312-cp312-linux_x86_64.whl size=21460597 sha256=7ac37c321ed246a0e192cfdca0516b66db701b64ffd9e6b98625eb834f2f54b8
  Stored in directory: /tmp/pip-ephem-wheel-cache-db92w710/wheels/bf/82/1d/663a33404c2a25ac83af673ef231e4bceb7dc6b55a288416a3
Successfully built fairseq
DEPRECATION: omegaconf 2.0.6 has a non

In [12]:
import os
import re

file_path = "/usr/local/lib/python3.12/dist-packages/fairseq/dataclass/configs.py"

# Buka file sistem yang rusak
with open(file_path, "r") as f:
    content = f.read()

# Pastikan fungsi 'field' dimuat
if "from dataclasses import field" not in content:
    content = "from dataclasses import field\n" + content

# Cari dan ganti semua format penulisan lama ke format baru Python 3.12
content = re.sub(r'(\w+):\s*([a-zA-Z0-9_]+)\s*=\s*\2\(\)', r'\1: \2 = field(default_factory=\2)', content)

# Simpan kembali file tersebut
with open(file_path, "w") as f:
    f.write(content)

print("Berhasil! Bug fairseq sudah diperbaiki paksa untuk Python 3.12.")

Berhasil! Bug fairseq sudah diperbaiki paksa untuk Python 3.12.


In [14]:
import os
import re

# Kita cari semua file konfigurasi hydra
folder_path = "/usr/local/lib/python3.12/dist-packages/hydra/conf"

for root, dirs, files in os.walk(folder_path):
    for file in files:
        if file.endswith(".py"):
            file_path = os.path.join(root, file)
            with open(file_path, "r") as f:
                content = f.read()

            # Tambahkan fungsi 'field' jika belum ada
            if "from dataclasses import field" not in content:
                content = "from dataclasses import field\n" + content

            # Ganti semua format lama ke default_factory
            new_content = re.sub(r'(\w+):\s*([a-zA-Z0-9_]+)\s*=\s*\2\(\)', r'\1: \2 = field(default_factory=\2)', content)

            # Simpan jika ada perubahan
            if new_content != content:
                with open(file_path, "w") as f:
                    f.write(new_content)
                print(f"Berhasil menambal: {file_path}")

print("Selesai! Bug Hydra sudah diperbaiki.")

Berhasil menambal: /usr/local/lib/python3.12/dist-packages/hydra/conf/__init__.py
Selesai! Bug Hydra sudah diperbaiki.


In [16]:
!sed -i 's/hydra_init()/# hydra_init()/g' /usr/local/lib/python3.12/dist-packages/fairseq/__init__.py
print("Sistem config fairseq yang rewel sudah dimatikan paksa!")

Sistem config fairseq yang rewel sudah dimatikan paksa!


In [18]:
import os
import re

# Targetkan seluruh folder fairseq
folder_path = "/usr/local/lib/python3.12/dist-packages/fairseq"
patched_count = 0

print("Memulai operasi sapu jagat untuk fairseq...")

for root, dirs, files in os.walk(folder_path):
    for file in files:
        if file.endswith(".py"):
            file_path = os.path.join(root, file)
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    content = f.read()

                # Coba cari format yang salah (Contoh: encoder: Config = Config() )
                if re.search(r'(\w+):\s*([a-zA-Z0-9_]+)\s*=\s*\2\(\)', content):
                    # Tambahkan impor dataclass jika belum ada
                    if "from dataclasses import field" not in content and "import field" not in content:
                        content = "from dataclasses import field\n" + content

                    # Ganti semua format lama ke default_factory
                    new_content = re.sub(r'(\w+):\s*([a-zA-Z0-9_]+)\s*=\s*\2\(\)', r'\1: \2 = field(default_factory=\2)', content)

                    with open(file_path, "w", encoding="utf-8") as f:
                        f.write(new_content)

                    print(f"Menambal file: {file}")
                    patched_count += 1
            except Exception as e:
                pass

print(f"Selesai! Berhasil memperbaiki {patched_count} file yang rusak.")

Memulai operasi sapu jagat untuk fairseq...
Menambal file: transformer_config.py
Menambal file: data2vec2.py
Menambal file: data2vec_text.py
Menambal file: infer.py
Menambal file: w2vu_generate.py
Menambal file: unpaired_audio_text.py
Menambal file: wav2vec_u.py
Selesai! Berhasil memperbaiki 7 file yang rusak.


In [21]:
import os
import re

folder_path = "/usr/local/lib/python3.12/dist-packages/fairseq"
patched_count = 0

print("Memulai operasi bedah tuntas untuk fairseq...")

for root, dirs, files in os.walk(folder_path):
    for file in files:
        if file.endswith(".py"):
            file_path = os.path.join(root, file)
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    content = f.read()

                # Pola ekstrim: Mencari semua class bermuka dua (Contoh: nama: Kelas = Kelas() )
                # dan memaksanya menjadi fungsi lambda agar Python 3.12 tidak marah.
                new_content = re.sub(
                    r'(:\s*)([A-Z][a-zA-Z0-9_]*)(\s*=\s*)\2\((.*?)\)',
                    r'\1\2 = field(default_factory=lambda: \2(\4))',
                    content
                )

                if new_content != content:
                    if "from dataclasses import field" not in new_content:
                        new_content = "from dataclasses import field\n" + new_content
                    with open(file_path, "w", encoding="utf-8") as f:
                        f.write(new_content)
                    print(f"Berhasil membedah file: {file}")
                    patched_count += 1
            except Exception as e:
                pass

print(f"Selesai! {patched_count} file berhasil diamputasi dari aturan Python 3.12.")

Memulai operasi bedah tuntas untuk fairseq...
Berhasil membedah file: prep_covost_data.py
Selesai! 1 file berhasil diamputasi dari aturan Python 3.12.


In [23]:
#@title Run inference in Command Line
#@markdown # Run inference in Command Line
#@markdown

#@markdown

#@markdown File and model Parameters
folder_input = "inputs" #@param {type:"string"}
store_dir = "results" #@param {type:"string"}
model_name = "multi_singing_librispeech_138" #@param ["vocals_238", "multi_singing_librispeech_138", "singing_librispeech_ft_iSRNet"]

#@markdown

#@markdown Common Parameters
use_overlapadd = "ola" #@param ["None", "ola", "ola_norm", "w2v", "w2v_chunk", "sf_chunk"]
separate_storage = False #@param {type:"boolean"}
skip_error = True #@param {type:"boolean"}
output_format = "wav" #@param ["wav", "flac", "mp3"]

#@markdown

#@markdown Advanced Parameters
vad_method = "spec" #@param ["spec", "webrtc"]
spectral_features = "mfcc" #@param ["mfcc", "spectral_centroid"]
ola_window_len = "0" #@param {type:"string"}
ola_hop_len = "0" #@param {type:"string"}
w2v_nth_layer_output = "0" #@param {type:"string"}
use_ema_model = True #@param {type:"boolean"}
mix_consistent_out = True #@param {type:"boolean"}
reorder_chunks = True #@param {type:"boolean"}

import os
import glob

MODEL_DIR = "checkpoint"
PRETRAINED_MODEL_DIR = "pretrained"
use_gpu = True

model_file = os.path.basename(glob.glob(os.path.join(MODEL_DIR, model_name, "*.pth"))[0])
target = model_file.replace(".pth", "")
exp_name = model_name
model_dir = MODEL_DIR
params = f"--target \"{target}\" --exp_name \"{exp_name}\" --model_dir \"{model_dir}\""
if use_gpu:
    params += " --use_gpu y"
else:
    params += " --use_gpu n"
if use_overlapadd != "None":
    params += f" --use_overlapadd {use_overlapadd}"
params += f" --vad_method {vad_method} --spectral_features {spectral_features} --w2v_ckpt_dir {PRETRAINED_MODEL_DIR} --w2v_nth_layer_output {w2v_nth_layer_output}"
if ola_window_len != "0":
    params += f" --ola_window_len {ola_window_len}"
if ola_hop_len != "0":
    params += f" --ola_hop_len {ola_hop_len}"
if use_ema_model:
    params += " --use_ema_model y"
else:
    params += " --use_ema_model n"
if mix_consistent_out:
    params += " --mix_consistent_out y"
else:
    params += " --mix_consistent_out n"
if reorder_chunks:
    params += " --reorder_chunks y"
else:
    params += " --reorder_chunks n"
if skip_error:
    params += " --skip_error y"
else:
    params += " --skip_error n"
if separate_storage:
    params += f" --separate_storage y"
else:
    params += f" --separate_storage n"
params += f" --output_format {output_format} --inference_data_dir \"{folder_input}\" --results_save_dir \"{store_dir}\""
print(params)

%cd /content/MedleyVox-Inference-WebUI
!python inference.py {params}

--target "vocals" --exp_name "multi_singing_librispeech_138" --model_dir "checkpoint" --use_gpu y --use_overlapadd ola --vad_method spec --spectral_features mfcc --w2v_ckpt_dir pretrained --w2v_nth_layer_output 0 --use_ema_model y --mix_consistent_out y --reorder_chunks y --skip_error y --separate_storage n --output_format wav --inference_data_dir "inputs" --results_save_dir "results"
/content/MedleyVox-Inference-WebUI
2026-03-16 07:24:24.746735: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773645864.768738   14464 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773645864.775157   14464 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:17